# Random search

In [ ]:
# DATA_DIR = DATA_DIR

TRAIN_DIR = DATA_DIR / 'train'
VALID_DIR = DATA_DIR / 'valid'
TEST_DIR = DATA_DIR / 'test'
NOISE_DIR = DATA_DIR / '_background_noise_'

print("Train:", len(list(TRAIN_DIR.rglob("*.wav"))))
print("Valid:", len(list(VALID_DIR.rglob("*.wav"))))

Train: 51486
Valid: 6828


In [ ]:
from pathlib import Path
import torch
import numpy as np
import random
import pandas as pd
import gc
import itertools

from data.utils import get_datasets, precompute_features
from data.dataset import CachedDataset
from data.transforms import FeatureConfig

from models.train import train, predict
from models.evaluate import evaluate
from models.transformer import Transformer
from models.cnn_transformer import CNNTransformer
from models.utils import set_seed

from plots import (
    plot_f1_training_curves,
    build_summary_df,
    plot_summary_table,
    plot_metrics_comparison,
    plot_f1_comparison,
    plot_confusion_matrix
)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print("Using device:", device)

Using device: cuda


## MEL

In [ ]:
train_ds, valid_ds, test_ds = get_datasets(
    data_format="mel",
    train_path=TRAIN_DIR,
    valid_path=VALID_DIR,
    test_path=TEST_DIR,
)

In [ ]:
CACHE_DIR = Path("/kaggle/working/cache_hparams_mel")

precompute_features(train_ds, CACHE_DIR / "train")
precompute_features(valid_ds, CACHE_DIR / "valid")
precompute_features(test_ds, CACHE_DIR / "test")

train_ds_cached = CachedDataset(CACHE_DIR / "train")
valid_ds_cached = CachedDataset(CACHE_DIR / "valid")
test_ds_cached = CachedDataset(CACHE_DIR / "test")

  0%|          | 0/51486 [00:00<?, ?it/s]

Saved 51486 samples to /kaggle/working/cache_hparams_mel/train


  0%|          | 0/6828 [00:00<?, ?it/s]

Saved 6828 samples to /kaggle/working/cache_hparams_mel/valid


  0%|          | 0/6865 [00:00<?, ?it/s]

Saved 6865 samples to /kaggle/working/cache_hparams_mel/test


In [ ]:
SEEDS = [0, 1, 2]
DEVICE = device
N_TRIALS = 10
EPOCHS = 15

In [ ]:
SEARCH_SPACE = {
    "lr": [1e-4, 2e-4, 3e-4, 5e-4, 1e-3],
    "dropout": [0.0, 0.1, 0.2, 0.3],
    "batch_size": [32, 64, 128],
}

In [ ]:
def sample_configs(n_trials=10, seed=0):
    all_configs = list(itertools.product(
        SEARCH_SPACE["lr"],
        SEARCH_SPACE["dropout"],
        SEARCH_SPACE["batch_size"]
    ))

    random.seed(seed)
    sampled = random.sample(all_configs, k=n_trials)

    return [
        {"lr": lr, "dropout": dropout, "batch_size": batch_size}
        for lr, dropout, batch_size in sampled
    ]

In [ ]:
def cnn_transformer_factory(repr_name="mel", dropout=0.1):
    n_features = 40 if repr_name == "mfcc" else 64

    return CNNTransformer(
        n_features=n_features,
        n_timesteps=101,
        num_classes=12,
        base_channels=32,
        d_model=128,
        nhead=4,
        num_layers=4,
        dropout=dropout,
        pooling="mean",
    )

In [ ]:
SAVE_PATH = "/kaggle/working/random_search_results.csv"

def random_search_experiment(n_trials=10, epochs=15):
    configs = sample_configs(n_trials=n_trials, seed=0)
    results = []

    for trial, cfg in enumerate(configs):
        lr = cfg["lr"]
        dropout = cfg["dropout"]
        batch_size = cfg["batch_size"]

        print(f"\nTrial {trial+1}/{n_trials}")
        print(f"lr={lr}, dropout={dropout}, batch={batch_size}")

        set_seed(0)  # one seed to see which parameters combinations are the best 

        model = cnn_transformer_factory(
            repr_name="mel",
            dropout=dropout,
        ).to(device)

        model, history = train(
            model,
            train_ds_cached,
            valid_ds_cached,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            device=device,
        )

        results.append({
            "trial": trial,
            "lr": lr,
            "dropout": dropout,
            "batch_size": batch_size,
            "best_valid_acc": max(history["valid_acc"]),
            "best_valid_macro_f1": max(history["valid_f1"]),
        })

        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)

        del model
        torch.cuda.empty_cache()
        gc.collect()

    return pd.DataFrame(results)

In [ ]:
df_results = random_search_experiment(n_trials=N_TRIALS, epochs=EPOCHS)
df_results.sort_values("best_valid_macro_f1", ascending=False)


Trial 1/10
lr=0.001, dropout=0.2, batch=32
Epoch 1/15 | Train Loss: 1.5284 | Valid Loss: 1.5490 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 2/15 | Train Loss: 1.5365 | Valid Loss: 1.5742 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 3/15 | Train Loss: 1.5347 | Valid Loss: 1.5629 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 4/15 | Train Loss: 1.5338 | Valid Loss: 1.5634 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 5/15 | Train Loss: 1.5312 | Valid Loss: 1.5620 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 6/15 | Train Loss: 1.5301 | Valid Loss: 1.5610 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 7/15 | Train Loss: 1.5287 | Valid Loss: 1.5619 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 8/15 | Train Loss: 1.5289 | Valid Loss: 1.5602 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 9/15 | Train Loss: 1.5277 | Valid Loss: 1.5600 | Valid Acc: 0.6182 | Valid Macro F1: 0.0637
Epoch 10/15 | Train Loss: 1.5276 | Valid Loss: 1.5606 | Valid Acc: 0.6182 

,trial,lr,dropout,batch_size,best_valid_acc,best_valid_macro_f1
6,6,0.0002,0.1,64,0.970562,0.955517
1,1,0.0003,0.0,32,0.969537,0.954923
8,8,0.0003,0.2,64,0.969977,0.953659
7,7,0.0003,0.2,128,0.967194,0.952273
3,3,0.0010,0.2,128,0.968658,0.952072
4,4,0.0003,0.0,128,0.968219,0.951368
9,9,0.0003,0.0,64,0.968219,0.950875
5,5,0.0001,0.0,128,0.962654,0.943555
0,0,0.0010,0.2,32,0.618190,0.063671
2,2,0.0010,0.0,32,0.618190,0.063671
